# Re-ranker training pipeline

Trains the baseline + all 6 learned re-ranking models (Simplex, RankSVM, Coordinate Ascent, MLP, REINFORCE, GBDT) on self-supervised labels derived from the catalog's category-path structure, and persists artifacts to `starter/reranker/artifacts/` for `starter/agent.py` to load at serving time.

This mirrors `training/train_all.py` (the reproducible CLI entry point) but walks through it interactively with plots. **This notebook does not touch `evaluator/local_evaluator.py`** -- all metrics computed here are offline diagnostics for model selection, not the official score (see the final cell).

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from starter.reranker.base import BaselineRanker, FEATURE_NAMES, rank_metrics
from training.common import load_agent, group_split
from training.evaluate import evaluate_predictions
from training.models.simplex_ranker import SimplexRanker
from training.models.ranksvm import RankSVMRanker
from training.models.coord_ascent import CoordinateAscentRanker
from training.models.mlp_ranker import MLPRanker
from training.models.reinforce_ranker import ReinforceRanker
from training.models.gbdt_ranker import GBDTRanker, grid_search
from starter.reranker.ensemble import threshold_union, minmax_normalize

ARTIFACTS_DIR = REPO_ROOT / "starter" / "reranker" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = REPO_ROOT / "data" / "reranker_training_data.npz"
RUN_GRID_SEARCH = False  # 27-config x 5-fold GBDT sweep -- slow, optional (see plan risk flags)
print("Feature order:", FEATURE_NAMES)

## 1. Data exploration -- validate the category-tree adaptation *before* trusting it

The catalog has no true product-to-product hierarchy, only a per-product `categories` path. Several features (`max_desc_sim`, `sibling_coherence`, `parent_sim`) depend on that path having realistic sibling/descendant/children populations. This is the single highest-risk assumption in the whole plan -- check it first.

In [ ]:
agent = load_agent(str(REPO_ROOT / "data" / "catalog.jsonl"))
cat_index = agent._cat_index
print(f"{len(agent._catalog):,} products, {len(cat_index.path_by_asin):,} with a category path, "
      f"{agent._doc_vecs.shape[0]:,} dense vectors")

depths = [len(p) for p in cat_index.path_by_asin.values()]
sibling_sizes = [len(v) for v in cat_index.siblings_by_path.values()]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(depths, bins=range(1, max(depths) + 2))
axes[0].set(title="Category path depth", xlabel="depth", ylabel="# products")
axes[1].hist(np.clip(sibling_sizes, 0, 200), bins=40)
axes[1].set(title="Same-full-path sibling group size (clipped at 200)", xlabel="group size", ylabel="# groups")
plt.tight_layout(); plt.show()

print("Median sibling-group size:", int(np.median(sibling_sizes)))
print("Groups of size 1 (no true siblings, will use parent-level fallback):",
      sum(1 for s in sibling_sizes if s <= 1), "/", len(sibling_sizes))

## 2. Generate self-supervised training labels

Runs `training/label_generation.py` as a subprocess (its own CLI is the reproducible entry point -- see the file's docstring for the exact rule set). Skips regeneration if the output file already exists; delete `data/reranker_training_data.npz` to force a rebuild.

In [ ]:
import subprocess

if not DATA_PATH.is_file():
    t0 = time.time()
    subprocess.run(
        [sys.executable, "-m", "training.label_generation",
         "--catalog", str(REPO_ROOT / "data" / "catalog.jsonl"),
         "--out", str(DATA_PATH), "--query-sample-frac", "0.15"],
        cwd=str(REPO_ROOT), check=True,
    )
    print(f"Label generation took {time.time()-t0:.0f}s")
else:
    print(f"Reusing existing {DATA_PATH}")

blob = np.load(DATA_PATH, allow_pickle=True)
X, y, groups = blob["X"], blob["y"], blob["groups"]
meta = blob["meta"][0]
print(meta)
print("Grade distribution:", {g: int((y == g).sum()) for g in sorted(set(y.tolist()))})

## 3. Train/test split (group-level, seed=42 -- one query's rows never split across sides)

In [ ]:
train_mask, test_mask = group_split(groups, test_frac=0.2, seed=42)
X_train, y_train, g_train = X[train_mask], y[train_mask], groups[train_mask]
X_test, y_test, g_test = X[test_mask], y[test_mask], groups[test_mask]
print(f"Train: {X_train.shape[0]} pairs / {len(np.unique(g_train))} groups")
print(f"Test : {X_test.shape[0]} pairs / {len(np.unique(g_test))} groups")

results = {}  # model_name -> metrics dict, filled in as each model trains below

## 4. Train all 7 models
### 4.0 Baseline (fixed weights, not learned)

In [ ]:
baseline = BaselineRanker()
baseline_scores = baseline.predict_scores(X_test)
results["baseline"] = evaluate_predictions(baseline_scores, y_test, g_test)
results["baseline"]

### 4.1 Simplex-constrained linear (SLSQP, w>=0, sum(w)=1)

In [ ]:
t0 = time.time()
simplex = SimplexRanker().fit(X_train, y_train, g_train)
simplex_scores = simplex.predict_scores(X_test)
results["simplex"] = evaluate_predictions(simplex_scores, y_test, g_test)
simplex.to_linear_ranker().save(ARTIFACTS_DIR / "simplex_weights.json")
print(f"{time.time()-t0:.1f}s -- weights: {dict(zip(FEATURE_NAMES, simplex.weights_.round(3)))}")
results["simplex"]

### 4.2 Pairwise RankSVM

In [ ]:
t0 = time.time()
ranksvm = RankSVMRanker().fit(X_train, y_train, g_train)
ranksvm_scores = ranksvm.predict_scores(X_test)
results["ranksvm"] = evaluate_predictions(ranksvm_scores, y_test, g_test)
ranksvm.to_linear_ranker().save(ARTIFACTS_DIR / "ranksvm_weights.json")
print(f"{time.time()-t0:.1f}s")
results["ranksvm"]

### 4.3 Coordinate Ascent (gradient-free, directly optimizes MRR)

In [ ]:
t0 = time.time()
coord_ascent = CoordinateAscentRanker().fit(X_train, y_train, g_train)
coord_scores = coord_ascent.predict_scores(X_test)
results["coord_ascent"] = evaluate_predictions(coord_scores, y_test, g_test)
coord_ascent.to_linear_ranker().save(ARTIFACTS_DIR / "coord_ascent_weights.json")
print(f"{time.time()-t0:.1f}s")
results["coord_ascent"]

### 4.4 MLP (11-32-16-1, BCE + 0.3*ApproxNDCG)

In [ ]:
t0 = time.time()
mlp = MLPRanker().fit(X_train, y_train, g_train)
mlp_scores = mlp.predict_scores(X_test)
results["mlp"] = evaluate_predictions(mlp_scores, y_test, g_test)
mlp.to_mlp_weights().save(ARTIFACTS_DIR / "mlp_ranker.npz", ARTIFACTS_DIR / "mlp_ranker.json")
print(f"{time.time()-t0:.1f}s, {len(mlp.history_)} epochs")

plt.figure(figsize=(6, 3))
plt.plot(mlp.history_)
plt.title("MLP validation BCE per epoch"); plt.xlabel("epoch"); plt.ylabel("val BCE")
plt.show()
results["mlp"]

### 4.5 REINFORCE (Plackett-Luce policy gradient, reward=NDCG@5, EMA baseline)

In [ ]:
t0 = time.time()
reinforce = ReinforceRanker().fit(X_train, y_train, g_train)
reinforce_scores = reinforce.predict_scores(X_test)
results["reinforce"] = evaluate_predictions(reinforce_scores, y_test, g_test)
reinforce.to_linear_ranker().save(ARTIFACTS_DIR / "reinforce_weights.json")
print(f"{time.time()-t0:.1f}s, {len(reinforce.reward_history_)} group updates")

rh = np.array(reinforce.reward_history_)
window = max(1, len(rh) // 200)
smoothed = np.convolve(rh, np.ones(window) / window, mode="valid")
plt.figure(figsize=(6, 3))
plt.plot(smoothed)
plt.title("REINFORCE reward (NDCG@5), smoothed"); plt.xlabel("group update"); plt.ylabel("reward")
plt.show()
results["reinforce"]

### 4.6 GBDT (LightGBM LambdaRank, label_gain=[0,1,3])

`RUN_GRID_SEARCH` (set in the setup cell) gates the optional 3x3x3 hyperparameter sweep -- it's genuinely optional; the fixed defaults below are what the source methodology shipped with.

In [ ]:
t0 = time.time()
if RUN_GRID_SEARCH:
    best_params, all_results = grid_search(X_train, y_train, g_train)
    print("Grid search best:", best_params)
    gbdt = GBDTRanker(n_estimators=500, learning_rate=best_params["learning_rate"],
                      num_leaves=best_params["num_leaves"],
                      min_data_in_leaf=best_params["min_data_in_leaf"]).fit(X_train, y_train, g_train)
else:
    gbdt = GBDTRanker().fit(X_train, y_train, g_train)
gbdt_scores = gbdt.predict_scores(X_test)
results["gbdt"] = evaluate_predictions(gbdt_scores, y_test, g_test)
gbdt.save(ARTIFACTS_DIR / "gbdtranker.txt", ARTIFACTS_DIR / "gbdtranker_feature_importances.json")
print(f"{time.time()-t0:.1f}s")

importances = pd.Series(gbdt.feature_importances_).sort_values(ascending=True)
importances.plot.barh(figsize=(6, 4), title="GBDT feature importance (gain)")
plt.tight_layout(); plt.show()
results["gbdt"]

## 5. Derive ensemble thresholds (re-derived on THIS dataset, not copied from the source project)

In [ ]:
from starter.reranker.base import ndcg_at_k

def ensemble_ndcg5(gbdt_s, mlp_s, y_arr, groups_arr, gbdt_t, mlp_t):
    scores_out = []
    for g in np.unique(groups_arr):
        mask = groups_arr == g
        g_y = y_arr[mask]
        if not np.any(g_y > 0):
            continue
        idx = np.nonzero(mask)[0]
        order = threshold_union(gbdt_s[idx], mlp_s[idx], list(range(len(idx))),
                                 gbdt_threshold=gbdt_t, mlp_threshold=mlp_t)
        scores_out.append(ndcg_at_k(g_y[order], 5))
    return float(np.mean(scores_out)) if scores_out else 0.0

best_thresh, best_ndcg5 = (0.72, 0.85), -1.0
sweep_rows = []
for gt in np.arange(0.5, 0.96, 0.05):
    for mt in np.arange(0.5, 0.96, 0.05):
        s = ensemble_ndcg5(gbdt_scores, mlp_scores, y_test, g_test, gt, mt)
        sweep_rows.append({"gbdt_threshold": round(float(gt), 2), "mlp_threshold": round(float(mt), 2), "ndcg@5": s})
        if s > best_ndcg5:
            best_ndcg5, best_thresh = s, (round(float(gt), 2), round(float(mt), 2))

import json
(ARTIFACTS_DIR / "ensemble_thresholds.json").write_text(
    json.dumps({"gbdt": best_thresh[0], "mlp": best_thresh[1], "val_ndcg@5": best_ndcg5}, indent=2))
print("Best thresholds:", best_thresh, "ndcg@5 =", round(best_ndcg5, 4))

## 6. Preview: baseline vs. each model, held-out split

In [ ]:
summary_df = pd.DataFrame(results).T.sort_values("ndcg@5", ascending=False)
summary_df

All artifacts are now saved to `starter/reranker/artifacts/`. Next: open `notebooks/model_comparison.ipynb` for a deeper side-by-side comparison, runtime-equivalence checks, and qualitative spot-checks -- then run `python -m evaluator.local_evaluator` (unmodified) to get the official TechnicalScore before deciding whether to ship `RERANK_ENABLED=1` by default.